# MLP paralelo — versión GPU en Google Colab

Este notebook corre la parte de GPU del proyecto en Colab:

1. Verifica que haya GPU y instala `numba-cuda`.
2. Obtiene el código del repositorio.
3. Corre los tests (incluidos los de GPU).
4. Mide el kernel CUDA separando cómputo y copias CPU↔GPU.
5. Entrena el MLP con el backend `gpu`.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.

## 1. GPU y dependencias

In [ ]:
# Debe mostrar una GPU (normalmente una Tesla T4) y la versión de CUDA del controlador
!nvidia-smi

In [ ]:
# Colab trae NumPy, Numba y scikit-learn. Solo falta el soporte CUDA para Numba.
# Usamos cu12 porque el controlador de Colab es de CUDA 12.x (ver nvidia-smi).
# NO instalamos requirements.txt aquí: cambiaría paquetes que Colab necesita.
!pip install -q "numba-cuda[cu12]"

## 2. Obtener el código

**Opción A (por defecto):** se clona el repositorio público indicado en `REPO_URL`.
**Opción B:** deja `REPO_URL` vacío y sube un `.zip` del proyecto (sin la carpeta `.venv`).

In [ ]:
import os, zipfile

REPO_URL = "https://github.com/davidatilano18/mlp-paralelo-mnist.git"  # vacío = subir un .zip
DIR = "/content/mlp-paralelo-mnist"

if not os.path.exists(DIR):
    if REPO_URL:
        !git clone -q {REPO_URL} {DIR}
    else:
        from google.colab import files
        subido = next(iter(files.upload()))
        with zipfile.ZipFile(subido) as z:
            z.extractall("/content")
        # Si el zip trae una carpeta con otro nombre, la renombramos
        if not os.path.exists(DIR):
            raiz = [d for d in os.listdir("/content") if os.path.isfile(f"/content/{d}/src/matmul.py")][0]
            os.rename(f"/content/{raiz}", DIR)

%cd {DIR}
!ls

## 3. Tests de correctitud

In [ ]:
# Con GPU, el backend "gpu" se registra solo y entra en TODOS los tests:
# comparación contra NumPy y equivalencia del entrenamiento entre backends.
!python -m pytest -q

In [ ]:
from src.matmul import BACKENDS
print("Backends disponibles:", list(BACKENDS))
assert "gpu" in BACKENDS, "No se detectó la GPU: revisa el tipo de entorno de ejecución"

## 4. Benchmark: cómputo vs. copias

Para cada tamaño se mide el tiempo **total** (copiar A y B a la GPU, kernel y copiar C de regreso),
el tiempo del **kernel** solo, y NumPy en CPU como referencia.

In [ ]:
!python -m experiments.bench_gpu --reps 5

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("results/bench_gpu.csv")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df.n, df.kernel_ms, "o-", label="GPU: solo kernel")
ax.plot(df.n, df.total_ms, "o-", label="GPU: total (con copias)")
ax.plot(df.n, df.numpy_ms, "s--", label="NumPy (CPU)")
ax.set(xscale="log", yscale="log", xlabel="n (matrices n × n)", ylabel="tiempo (ms)",
       title="Matmul en GPU: cómputo vs. transferencias")
ax.legend(); ax.grid(True, which="both", alpha=.3)
plt.show()
df

## 5. Entrenar el MLP en la GPU

In [ ]:
# Misma red y misma semilla que en CPU: la precisión debe ser prácticamente igual.
!python -m experiments.entrenar --backend gpu --epocas 10

## 6. ¿Qué observar?

- **Matrices grandes:** el kernel es mucho más rápido que la CPU, porque miles de hilos trabajan a la vez.
- **Matrices chicas** (las del MLP con lotes de 64): las copias CPU↔GPU por PCIe dominan el tiempo.
  Por eso entrenar con `gpu` puede ser *más lento* que con `bloques` en CPU.
- **Cómo se mejoraría:** dejar los pesos y las activaciones en la GPU durante todo el entrenamiento
  y copiar solo los lotes de entrada. Eso requiere que todas las operaciones de la red
  (ReLU, softmax, actualización de pesos) también corran en la GPU, no solo el matmul.